# Exercise C: Operational Scenario — Day-to-Day ML Operations

**Snowpark ML Fundamentals — Day 3 Morning**

---

## Business Scenario

NCL's guest churn prediction model has been in production for 3 months.
Three operational tasks have landed on your desk:

| Task | Situation |
|------|----------|
| **1. Retrain** | New guest data has arrived — retrain and compare against production |
| **2. Update Features** | Business wants a new feature added to the Feature Store |
| **3. Troubleshoot** | The nightly scoring pipeline is failing — diagnose and fix |

## Rules
- Fill in every `# TODO:` — each requires **1–5 lines of code**
- Run the **Validation** cells to check your work
- **3 checkpoints** where the instructor reviews progress
- Estimated time: **45 minutes**

---

In [ ]:
# --- Setup (given — just run this cell) ---
import sys
sys.path.insert(0, '../../../src')

import logging
import warnings

import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

logger = logging.getLogger("module_23")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

from snowflake.snowpark import functions as F

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.loader import split_data, explore_dataframe
from snowpark_fundamentals.modeling.trainer import train_model, predict
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier
from snowpark_fundamentals.feature_store.entities import (
    setup_feature_store, create_customer_entity, register_entity,
)
from snowpark_fundamentals.feature_store.feature_data import (
    create_customer_transactions_dataset,
    create_customer_interactions_dataset,
    create_rfm_features,
    create_behavioral_features,
)
from snowpark_fundamentals.feature_store.feature_views import (
    create_external_feature_view, register_feature_view, get_feature_view,
)
from snowpark_fundamentals.feature_store.training_sets import (
    create_spine_dataframe, generate_training_set,
)
from snowpark_fundamentals.registry.model_registry import (
    get_registry, log_model, delete_model,
    compare_model_versions, set_model_alias,
    get_model_by_alias, set_model_comment, set_default_version,
    get_model_metrics, list_versions,
    load_model_and_predict,
)

session = create_session(env_path='../../../.env')

# Schema isolation
STUDENT_NAME = "YOUR_NAME"  # <-- CHANGE THIS
session.sql("USE ROLE MLDS_ROLE_D").collect()
session.sql("USE DATABASE MLDS_D").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {STUDENT_NAME}_ONSITE_TRAINING").collect()
session.sql(f"USE SCHEMA {STUDENT_NAME}_ONSITE_TRAINING").collect()
logger.info(f"Working in schema: {STUDENT_NAME}_ONSITE_TRAINING")

⏳ **Note:** The setup cell below takes 1–2 minutes on a Size-S warehouse. This is normal — it builds your simulated production environment with existing model, feature views, and training data. Run it once and proceed to Task 1.

In [ ]:
# --- Pre-build production state (simulates 3-month-old deployment) ---
# Generate "original" training data
transactions_df = create_customer_transactions_dataset(session, n_rows=8000, seed=42)
interactions_df = create_customer_interactions_dataset(session, n_rows=16000, seed=42)

# Build features and Feature Store
rfm_df = create_rfm_features(session)
behavioral_df = create_behavioral_features(session)

fs = setup_feature_store(session)
entity = create_customer_entity(desc='NCL guest entity')
entity = register_entity(fs, entity)

rfm_fv = create_external_feature_view(
    name=f"{STUDENT_NAME}_OPS_RFM",
    entities=[entity], feature_df=rfm_df,
    desc="RFM features V1", timestamp_col="_FEATURE_TIMESTAMP",
)
rfm_fv = register_feature_view(fs, rfm_fv, version='V1', overwrite=True)

behavioral_fv = create_external_feature_view(
    name=f"{STUDENT_NAME}_OPS_BEHAVIORAL",
    entities=[entity], feature_df=behavioral_df,
    desc="Behavioral features V1", timestamp_col="_FEATURE_TIMESTAMP",
)
behavioral_fv = register_feature_view(fs, behavioral_fv, version='V1', overwrite=True)

# Generate training set and train production model
spine_df = create_spine_dataframe(session, entity_table='CUSTOMER_RFM_FEATURES')
rfm_fv_reg = get_feature_view(fs, f"{STUDENT_NAME}_OPS_RFM", 'V1')
behavioral_fv_reg = get_feature_view(fs, f"{STUDENT_NAME}_OPS_BEHAVIORAL", 'V1')
training_set = generate_training_set(
    fs, spine_df, features=[rfm_fv_reg, behavioral_fv_reg],
    name=f'{STUDENT_NAME}_OPS_TRAINING',
)
training_df = training_set.read.to_snowpark_dataframe()

# Create labels and train
FEATURE_COLS = [
    'DAYS_SINCE_LAST_ORDER', 'ORDERS_30D', 'ORDERS_90D', 'ORDERS_365D',
    'ORDERS_TOTAL', 'SPEND_30D', 'SPEND_90D', 'SPEND_365D', 'SPEND_TOTAL',
    'AVG_ORDER_VALUE', 'TOTAL_ITEMS', 'AVG_ITEMS_PER_ORDER',
    'TOTAL_PAGE_VIEWS', 'TOTAL_CLICKS', 'TOTAL_SUPPORT_TICKETS',
    'TOTAL_EMAIL_ENGAGEMENTS', 'INTERACTIONS_30D', 'SUPPORT_TICKETS_30D',
    'INTERACTIONS_90D', 'DAYS_SINCE_LAST_INTERACTION', 'AVG_DURATION_SECONDS',
]
LABEL_COL = 'CHURNED'

labeled_df = (
    training_df
    .with_column("_NOISE", F.uniform(0.0, 1.0, F.random(42)))
    .with_column(
        LABEL_COL,
        F.when(
            (F.col("SUPPORT_TICKETS_30D") >= 2) & (F.col("ORDERS_30D") <= 1)
            & (F.col("_NOISE") < F.lit(0.85)),
            F.lit(1)
        ).when(
            (F.col("INTERACTIONS_30D") < 3) & (F.col("SPEND_30D") < F.lit(500))
            & (F.col("_NOISE") < F.lit(0.15)),
            F.lit(1)
        ).when(F.col("_NOISE") < F.lit(0.03), F.lit(1))
        .otherwise(F.lit(0))
    )
    .drop("_NOISE")
)

all_data, holdout_df = split_data(labeled_df.select(FEATURE_COLS + [LABEL_COL]), train_ratio=0.8)
train_df, test_df = split_data(all_data, train_ratio=0.75)

production_model = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='xgboost', model_params={'n_estimators': 80, 'max_depth': 5},
)
prod_preds = predict(production_model, test_df)
prod_metrics = evaluate_binary_classifier(prod_preds, LABEL_COL, 'PREDICTION')

current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

MODEL_NAME = f"{STUDENT_NAME}_CHURN_OPS"
try:
    delete_model(registry, MODEL_NAME)
except Exception:
    pass

log_model(registry, production_model.to_xgboost(), MODEL_NAME, 'V1',
          sample_input=test_df.select(FEATURE_COLS).limit(10), metrics=prod_metrics)
set_model_alias(registry, MODEL_NAME, 'V1', 'production')
set_default_version(registry, MODEL_NAME, 'V1')
set_model_comment(registry, MODEL_NAME,
    "V1: Original production model. XGBoost n_estimators=80, max_depth=5. Trained on 8K transactions.",
    version_name='V1')

logger.info(f"Production state ready: {MODEL_NAME} V1 with 'production' alias")
logger.info(f"Production metrics: {prod_metrics}")
logger.info(f"Holdout set reserved: {holdout_df.count()} rows")

---

## Task 1: Retrain with New Data (12 min)

**Situation:** 3 months of new guest data have arrived (2,000 additional transactions).
Retrain the model on the combined dataset and evaluate whether
the new model should replace the current production version.

In [ ]:
# --- New data has arrived (simulate 3 months of new transactions) ---
new_transactions = create_customer_transactions_dataset(
    session, table_name='NEW_CUSTOMER_TRANSACTIONS',
    n_rows=2000, seed=99,
)
logger.info(f"New transactions arrived: {new_transactions.count()} rows")
logger.info("(These represent 3 months of new guest booking activity)")

In [ ]:
# TODO A1: Compute fresh RFM features from the updated data
# The create_rfm_features() function reads from CUSTOMER_TRANSACTIONS by default.
# First, combine old + new transactions into one table, then compute features.
# Hint: Use session.sql() to INSERT INTO or UNION the new data, or just
#   call create_rfm_features() which will use whatever is in CUSTOMER_TRANSACTIONS

# Combine old and new data
session.sql("""
    CREATE OR REPLACE TABLE CUSTOMER_TRANSACTIONS AS
    SELECT * FROM CUSTOMER_TRANSACTIONS
    UNION ALL
    SELECT * FROM NEW_CUSTOMER_TRANSACTIONS
""").collect()
logger.info(f"Combined transactions: {session.table('CUSTOMER_TRANSACTIONS').count()} rows")

# Recompute features
updated_rfm = ____

logger.info(f"Updated RFM features: {updated_rfm.count()} customers")

In [ ]:
# TODO A2: Generate a new training set from the Feature Store with updated features
# Steps:
#   1. Register updated RFM as a new feature view version (V2)
#   2. Create spine from the updated features
#   3. Generate training set
# Hint: register_feature_view(fs, ..., version='V2', overwrite=True)

# Register updated feature view
updated_rfm_fv = create_external_feature_view(
    name=f"{STUDENT_NAME}_OPS_RFM",
    entities=[entity], feature_df=updated_rfm,
    desc="RFM features V2 \u2014 with 3 months new data",
    timestamp_col="_FEATURE_TIMESTAMP",
)
updated_rfm_fv = register_feature_view(fs, updated_rfm_fv, version='V2', overwrite=True)

# Create spine and generate training set
spine_df = ____
updated_rfm_fv_reg = get_feature_view(fs, f"{STUDENT_NAME}_OPS_RFM", 'V2')
behavioral_fv_reg = get_feature_view(fs, f"{STUDENT_NAME}_OPS_BEHAVIORAL", 'V1')

new_training_set = ____

new_training_df = new_training_set.read.to_snowpark_dataframe()
logger.info(f"New training set: {new_training_df.count()} rows, {len(new_training_df.columns)} cols")

In [ ]:
# --- Label creation for new training data (same logic as original) ---
new_labeled_df = (
    new_training_df
    .with_column("_NOISE", F.uniform(0.0, 1.0, F.random(42)))
    .with_column(
        LABEL_COL,
        F.when(
            (F.col("SUPPORT_TICKETS_30D") >= 2) & (F.col("ORDERS_30D") <= 1)
            & (F.col("_NOISE") < F.lit(0.85)),
            F.lit(1)
        ).when(
            (F.col("INTERACTIONS_30D") < 3) & (F.col("SPEND_30D") < F.lit(500))
            & (F.col("_NOISE") < F.lit(0.15)),
            F.lit(1)
        ).when(F.col("_NOISE") < F.lit(0.03), F.lit(1))
        .otherwise(F.lit(0))
    )
    .drop("_NOISE")
)
new_train_df, new_test_df = split_data(new_labeled_df.select(FEATURE_COLS + [LABEL_COL]))
logger.info(f"New train: {new_train_df.count()}, New test: {new_test_df.count()}")

In [ ]:
# TODO A3: Train a new model on the updated data
# Hint: train_model() with improved params (n_estimators=150, max_depth=7)

new_model = ____

new_preds = predict(new_model, new_test_df)
new_metrics = evaluate_binary_classifier(new_preds, LABEL_COL, 'PREDICTION')
logger.info(f"New model metrics: {new_metrics}")

In [ ]:
# TODO A4: Champion/Challenger \u2014 compare new model against production
# Steps:
#   1. Load the current production model by alias
#   2. Score holdout with champion
#   3. Score holdout with challenger
#   4. Print both F1 scores
# Hint: include LABEL_COL in the scoring input so evaluation still works

# 1. Load champion
champion = ____
champion_preds = champion.run(
    holdout_df.select(FEATURE_COLS + [LABEL_COL]),
    function_name='predict',
)
champion_columns = {str(col).strip('\"').upper(): str(col) for col in champion_preds.columns}
champion_prediction_col = champion_columns.get('PREDICTION') or champion_columns.get('OUTPUT_FEATURE_0')
if champion_prediction_col != 'PREDICTION':
    champion_preds = champion_preds.with_column_renamed(champion_prediction_col, 'PREDICTION')
champion_metrics = evaluate_binary_classifier(champion_preds, LABEL_COL, 'PREDICTION')

# 2. Evaluate challenger on same holdout
challenger_preds = predict(new_model, holdout_df)
challenger_metrics = evaluate_binary_classifier(challenger_preds, LABEL_COL, 'PREDICTION')

# 3. Compare
logger.info(f"Champion (V1) holdout F1: {champion_metrics['f1_score']:.4f}")
logger.info(f"Challenger       holdout F1: {challenger_metrics['f1_score']:.4f}")
improvement = challenger_metrics['f1_score'] - champion_metrics['f1_score']
logger.info(f"Improvement: {improvement:+.4f}")

In [ ]:
# --- Validation Task 1 ---
assert updated_rfm.count() > 0, "A1: updated features should have rows"
assert new_training_df.count() > 0, "A2: new training set should have rows"
assert isinstance(new_metrics, dict), "A3: new_metrics should be a dict"
assert isinstance(champion_metrics, dict), "A4: champion_metrics should be a dict"
logger.info("Task 1: All validations passed!")

---

### \u23f8 Checkpoint 1 \u2014 Instructor reviews Task 1

---

## Task 2: Update Feature Store with New Feature (10 min)

**Situation:** The business team wants a new feature: **support ticket velocity** \u2014
the rate of support tickets in the last 30 days relative to the 3-month average.

A velocity > 1.0 means tickets are increasing; < 1.0 means decreasing.

In [ ]:
# TODO B1: Compute SUPPORT_TICKET_VELOCITY
# Formula: SUPPORT_TICKETS_30D / NULLIF(INTERACTIONS_90D / 3.0, 0)
#   This gives the 30-day rate relative to the monthly average over 90 days.
#   >1 = increasing rate, <1 = decreasing rate
# Hint: Use session.sql() to SELECT from the behavioral features table
#   and add the computed column

enhanced_features = session.sql("""
    SELECT b.*,
        ____  AS SUPPORT_TICKET_VELOCITY
    FROM CUSTOMER_BEHAVIORAL_FEATURES b
""")

logger.info(f"Enhanced features: {enhanced_features.count()} rows")
enhanced_features.select(
    'CUSTOMER_ID', 'SUPPORT_TICKETS_30D', 'INTERACTIONS_90D',
    'SUPPORT_TICKET_VELOCITY'
).show(5)

In [ ]:
# TODO B2: Register updated feature view as V2
# Hint: First call create_external_feature_view(), then pass the result
#   to register_feature_view() with version='V2'. Reassign to same variable.

updated_beh_fv = ____  # create_external_feature_view(...)

updated_beh_fv = ____  # register_feature_view(fs, updated_beh_fv, ...)

logger.info("Updated behavioral feature view registered as V2.")

In [ ]:
# TODO B3: Verify the new feature view has the SUPPORT_TICKET_VELOCITY column
# Hint: get_feature_view(fs, name, 'V2')

verified_fv = ____

logger.info(f"Feature view: {verified_fv.name} V2")
logger.info(f"Verified \u2014 new feature view is available in the Feature Store.")

In [ ]:
# --- Validation Task 2 ---
assert 'SUPPORT_TICKET_VELOCITY' in enhanced_features.columns, \
    "B1: SUPPORT_TICKET_VELOCITY column should exist"
logger.info("Task 2: All validations passed!")

---

### \u23f8 Checkpoint 2 \u2014 Instructor reviews Task 2

---

## Task 3: Promote & Troubleshoot (13 min)

Two sub-tasks:
1. If the challenger model is better, promote it to production
2. The nightly scoring stored procedure is failing \u2014 find and fix the bug

In [ ]:
# TODO C1: If challenger is better, register as V2 and promote to production
# Hint: Use log_model(), set_model_alias(), set_default_version(), set_model_comment()

if challenger_metrics['f1_score'] > champion_metrics['f1_score']:
    # Register new version
    ____

    # Move production alias and default version
    ____
    ____

    # Document the promotion
    ____

    logger.info(f"New model promoted to production as V2")
    logger.info(f"  Champion F1: {champion_metrics['f1_score']:.4f}")
    logger.info(f"  Challenger F1: {challenger_metrics['f1_score']:.4f}")
else:
    logger.info("Champion retained \u2014 challenger did not improve F1.")

### The Nightly Scoring Pipeline Is Failing

While you were retraining, the on-call engineer reports that the nightly
scoring stored procedure is throwing an error. Your job: find the bug and fix it.

In [ ]:
# --- The broken stored procedure ---
broken_sp = f"""
CREATE OR REPLACE PROCEDURE {STUDENT_NAME}_NIGHTLY_SCORING(
    MODEL_NAME VARCHAR,
    OUTPUT_TABLE VARCHAR
)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'snowflake-ml-python')
HANDLER = 'run'
AS $$
def run(session, model_name: str, output_table: str) -> str:
    from snowflake.ml.registry import Registry

    # Load model
    registry = Registry(session=session)
    model_ref = registry.get_model(model_name)
    mv = model_ref.default

    # Read features -- BUG: wrong table name!
    features_df = session.table('CUSTOOMER_RFM_FEEATURES')

    feature_cols = [
        'DAYS_SINCE_LAST_ORDER', 'ORDERS_30D', 'ORDERS_90D',
        'ORDERS_TOTAL', 'SPEND_30D', 'SPEND_90D', 'SPEND_TOTAL',
        'AVG_ORDER_VALUE', 'TOTAL_ITEMS',
    ]

    # Score
    predictions = mv.run(features_df[feature_cols], function_name='predict')
    predictions.write.mode('overwrite').save_as_table(output_table)
    count = session.table(output_table).count()
    return f'Scored {{count}} rows into {{output_table}}'
$$;
"""
session.sql(broken_sp).collect()

# Try calling it \u2014 this will fail
logger.info("Calling the nightly scoring procedure...")
try:
    session.sql(f"""
        CALL {STUDENT_NAME}_NIGHTLY_SCORING('{MODEL_NAME}', '{STUDENT_NAME}_NIGHTLY_PREDICTIONS')
    """).collect()
except Exception as e:
    logger.info(f"\nERROR: {e}")
    logger.info("\n^ Read the error message carefully. The bug is in the stored procedure.")

In [ ]:
# TODO C3: Fix the stored procedure \u2014 the current version builds the wrong scoring dataset
# It points at the wrong table name and omits the behavioral features
# required by the production model signature.
# Hint: join CUSTOMER_RFM_FEATURES to CUSTOMER_BEHAVIORAL_FEATURES on CUSTOMER_ID
# and score with the same full feature list used during training.

fixed_sp = f"""
CREATE OR REPLACE PROCEDURE {STUDENT_NAME}_NIGHTLY_SCORING(
    MODEL_NAME VARCHAR,
    OUTPUT_TABLE VARCHAR
)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'snowflake-ml-python')
HANDLER = 'run'
AS $$
def run(session, model_name: str, output_table: str) -> str:
    from snowflake.ml.registry import Registry

    registry = Registry(session=session)
    model_ref = registry.get_model(model_name)
    mv = model_ref.default

    feature_cols = [
        'DAYS_SINCE_LAST_ORDER', 'ORDERS_30D', 'ORDERS_90D', 'ORDERS_365D',
        'ORDERS_TOTAL', 'SPEND_30D', 'SPEND_90D', 'SPEND_365D', 'SPEND_TOTAL',
        'AVG_ORDER_VALUE', 'TOTAL_ITEMS', 'AVG_ITEMS_PER_ORDER',
        'TOTAL_PAGE_VIEWS', 'TOTAL_CLICKS', 'TOTAL_SUPPORT_TICKETS',
        'TOTAL_EMAIL_ENGAGEMENTS', 'INTERACTIONS_30D', 'SUPPORT_TICKETS_30D',
        'INTERACTIONS_90D', 'DAYS_SINCE_LAST_INTERACTION', 'AVG_DURATION_SECONDS',
    ]

    features_df = session.sql(____)

    predictions = mv.run(features_df[['CUSTOMER_ID'] + feature_cols], function_name='predict')
    normalized_columns = {{str(col).strip('\"').upper(): str(col) for col in predictions.columns}}
    prediction_col = normalized_columns.get('PREDICTION') or normalized_columns.get('OUTPUT_FEATURE_0')
    if prediction_col != 'PREDICTION':
        predictions = predictions.with_column_renamed(prediction_col, 'PREDICTION')

    predictions.write.mode('overwrite').save_as_table(output_table)
    count = session.table(output_table).count()
    return f'Scored {{count}} rows into {{output_table}}'
$$;
"""
session.sql(fixed_sp).collect()
logger.info("Fixed stored procedure created.")

In [ ]:
# TODO C4: Call the fixed stored procedure and verify it works
# Hint: session.sql("CALL ...").collect()

result = ____

logger.info(f"Result: {result[0][0]}")
logger.info("\nPredictions sample:")
session.table(f'{STUDENT_NAME}_NIGHTLY_PREDICTIONS').show(5)

In [ ]:
# --- Validation Task 3 ---
nightly_count = session.table(f'{STUDENT_NAME}_NIGHTLY_PREDICTIONS').count()
assert nightly_count > 0, "C3/C4: nightly predictions table should have rows"
logger.info(f"Task 3: All validations passed! ({nightly_count} rows scored)")

logger.info("\n" + "=" * 50)
logger.info("EXERCISE C COMPLETE!")
logger.info("=" * 50)
logger.info("\nYou've handled 3 real operational tasks:")
logger.info("  1. Retrained with new data + champion/challenger comparison")
logger.info("  2. Added a new feature to the Feature Store")
logger.info("  3. Diagnosed and fixed a broken scoring pipeline")

---

### \u23f8 Checkpoint 3 \u2014 Instructor reviews Task 3

## Training Complete!

Over 3 days you've covered the full ML lifecycle in Snowflake:

| Day | Topics |
|-----|--------|
| **1** | Experiment tracking, validation, champion/challenger, monitoring |
| **2** | UDFs, window functions, stored procedures, Pandas migration |
| **3** | End-to-end classification, operational workflows, troubleshooting |

These patterns apply directly to NCL's guest analytics workloads.

In [ ]:
# --- Cleanup ---
try:
    delete_model(registry, MODEL_NAME)
    session.sql("DROP TABLE IF EXISTS NEW_CUSTOMER_TRANSACTIONS").collect()
    session.sql(f"DROP TABLE IF EXISTS {STUDENT_NAME}_NIGHTLY_PREDICTIONS").collect()
    logger.info("Cleaned up.")
except Exception as e:
    logger.warning("Cleanup note: %s", e)

In [ ]:
session.close()